In [0]:
%sql
-- Populate with unique products from Silver layer
INSERT OVERWRITE TABLE gold_db.dim_products
SELECT
    -- Generate a unique product ID from sku + brand
    md5(concat(sku, brand))              AS product_id,
    sku                                  AS sku_code,
    max_by(name, scraped_at)             AS product_name,
    brand,
    max_by(category_path, scraped_at)    AS category_path,
    max_by(categories, scraped_at)       AS categories,
    max_by(url, scraped_at)              AS product_url,
    MIN(scraped_at)                      AS first_seen_date,
    MAX(scraped_at)                      AS last_seen_date
FROM gold_db.silver_products
GROUP BY sku, brand;

In [0]:
# %sql
# select *
# from gold_db.dim_products

In [0]:
%sql
-- Populate price history from Silver layer
MERGE INTO gold_db.fct_prices AS target
USING (
    SELECT 
        md5(concat(sku, CAST(scraped_at AS STRING))) AS price_id,
        md5(concat(sku, brand)) AS product_id,
        DATE(scraped_at) AS scraped_date,
        scraped_at,
        price
    FROM gold_db.silver_products
) AS source
ON target.price_id = source.price_id
WHEN NOT MATCHED THEN INSERT *;

In [0]:
# %sql
# select *
# from gold_db.fct_prices

In [0]:
# %sql
# -- query for the dashboard 
# SELECT sku_code, product_name, brand, categories, scraped_date, scraped_at, price
# FROM (
#     SELECT 
#         d.sku_code, d.product_name, d.brand, d.categories,
#         f.scraped_date, f.scraped_at, f.price,
#         ROW_NUMBER() OVER (
#             PARTITION BY f.product_id, f.scraped_date 
#             ORDER BY f.scraped_at DESC
#         ) AS rn
#     FROM gold_db.dim_products d
#     JOIN gold_db.fct_prices f USING (product_id)
# )
# WHERE rn = 1
# ORDER BY sku_code, scraped_date
